<a href="https://colab.research.google.com/github/ckrickyh/pythonTools/blob/main/TMCP_F1_DataSummarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geopandas as gpd
import pandas as pd
import json
import xlwings as xw
import zipfile
import pathlib
from pathlib import Path
import re

In [ ]:
#invalidtext = ['draft', 'unsumitted']
#includetext = ['zip']

invalidtext_org = ['draft', 'unsumitted', 'form2']
includetext_org = ['Submission'] #['ok'] #Folder 放zip既關鍵名

#===================================================關鍵字強行細階化
invalidtext = []
includetext = []

invalidtext = [text.lower() for text in invalidtext_org]
includetext = [text.lower() for text in includetext_org]

#===================================================
#folderpath = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4'
folderpath = input('Paste fld path: ')

Paste fld path:  H:\LU\TRAM 10 Form 1 (2024)\Submission Report


In [ ]:
fldcat = input('Folder category [1: return xl in each folder; 2: return xl in parent fld; 3: read zip within zip, return xl to parent fld] - ')

fldcatList = [1,2,3]

if fldcat not in fldcatList:
    fldcat = 1
print(fldcat)

Folder category [1: return xl in each folder; 2: return xl in parent fld; 3: read zip within zip, return xl to parent fld] -  2


1


# TMCP database csv

In [ ]:
tmcppath = r'C:\TMCP\CSVUnZIPTemp'

#create dataframe blank in dic
d={}

count = 0

tmcp_list = ['TBL_TREE_SPECIES.csv', 'TBL_TRIAGE_COLOUR.csv', 'TBL_MITIGATION_MEASURE.csv', 'TBL_TREE_CONDITION.csv', 'TBL_REMEDIAL_ACTION.csv', 'TBL_TREE_STATUS.csv']
lookup_list = ['TreeSpecies', 'TriageColour', 'MitigationMeasure', 'OverallConditions', 'RemedialAction', 'TreeStatus']

for i in tmcp_list:
    csvpath = Path('/').joinpath(tmcppath, i)

    d[i] = pd.read_csv(csvpath, sep='\t')

    #add column to dymanic dataframe
    d[i].insert(0, 'MasterTypeCode', lookup_list[count])

    #=====
    df_tmcp = d[i]
    df_tmcp.rename(columns={df_tmcp.columns[3]: 'Code', df_tmcp.columns[4]:'info_eng', df_tmcp.columns[5]:'info_chi'},inplace=True)

    if count == 0:
        df_tmcpA = pd.DataFrame(columns = df_tmcp.columns.tolist())
    df_tmcpA = pd.concat([df_tmcpA, df_tmcp])

    count = count +1

In [ ]:
df_tmcp

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,TreeStatus,1,1,1,Old and Valuable Tree,古樹名木,10/15/2019 18:52,03/04/2025 13:46,Y,1
1,TreeStatus,2,2,2,Stonewall Tree,石牆樹,10/15/2019 18:52,03/04/2025 13:46,Y,2
2,TreeStatus,3,3,3,Brown Root Rot Disease Infected,受褐根病感染,10/15/2019 18:52,03/04/2025 13:46,Y,3
3,TreeStatus,4,4,4,Mature Tree,成齡樹,01/12/2024 01:04,03/04/2025 13:46,N,4
4,TreeStatus,5,5,5,Other Trees,其他樹木,10/15/2019 18:52,03/04/2025 13:46,Y,7
5,TreeStatus,6,6,6,Tree in Confined Site,擠迫地點的樹木,07/19/2023 01:40,03/04/2025 13:46,Y,6
6,TreeStatus,7,7,7,Large Tree(DBH >= 500mm or overall height >= 9m),大樹(胸徑≥500毫米或高度≥9米),07/19/2023 02:32,03/04/2025 13:46,Y,5


In [ ]:
d['TBL_TREE_SPECIES.csv'].head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,TreeSpecies,818,1,1,Acacia auriculiformis,耳果相思(耳葉相思),03/06/2020 11:17,03/04/2025 13:46,Y,1
1,TreeSpecies,819,2,2,Acacia confusa,台灣相思,11/24/2015 14:10,03/04/2025 13:46,Y,2
2,TreeSpecies,820,3,3,Acacia dealbata,銀荊,11/24/2015 14:10,03/04/2025 13:46,Y,3
3,TreeSpecies,1541,725,725,Acacia farnesiana,金合歡,02/15/2017 12:06,03/04/2025 13:46,Y,4
4,TreeSpecies,821,4,4,Acacia holosericea,絹毛相思,11/24/2015 14:10,03/04/2025 13:46,Y,5


In [ ]:
d['TBL_TRIAGE_COLOUR.csv'].head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,TriageColour,5,4,4,Orange,橙,07/19/2023 01:39,03/04/2025 13:46,Y,3
1,TriageColour,2,3,3,Black,黑,10/15/2019 18:59,03/04/2025 13:46,Y,1
2,TriageColour,3,2,2,Red,紅,10/15/2019 18:59,03/04/2025 13:46,Y,2
3,TriageColour,4,1,1,Yellow,黃,10/15/2019 18:59,03/04/2025 13:46,Y,4
4,TriageColour,1,0,0,NIL (no classification),無,10/15/2019 18:59,03/04/2025 13:46,Y,5


In [ ]:
d['TBL_MITIGATION_MEASURE.csv'].head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,MitigationMeasure,2,1,1,Crown cleaning,清理樹冠,10/25/2019 14:25,03/04/2025 13:46,Y,1
1,MitigationMeasure,3,2,2,Crown thinning,樹冠疏理,10/25/2019 14:28,03/04/2025 13:46,Y,2
2,MitigationMeasure,4,3,3,Crown reduction,縮減樹冠,10/25/2019 14:28,03/04/2025 13:46,Y,3
3,MitigationMeasure,5,4,4,Structural Pruning,結構修剪,10/25/2019 14:28,03/04/2025 13:46,Y,4
4,MitigationMeasure,6,5,5,Crown raising,樹冠提升,10/25/2019 14:28,03/04/2025 13:46,Y,5


In [ ]:
d['TBL_TREE_CONDITION.csv'].head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,OverallConditions,1,1,1,Normal,正常,10/15/2019 18:54,03/04/2025 13:46,Y,1
1,OverallConditions,2,2,2,Fair,一般,10/15/2019 18:54,03/04/2025 13:46,Y,2
2,OverallConditions,3,3,3,Poor,差,10/15/2019 18:54,03/04/2025 13:46,Y,3
3,OverallConditions,4,4,4,Very Poor,十分差,10/15/2019 18:54,03/04/2025 13:46,Y,4
4,OverallConditions,5,5,5,Dead Tree,死樹,10/15/2019 18:54,03/04/2025 13:46,Y,5


In [ ]:
d['TBL_REMEDIAL_ACTION.csv'].head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,RemedialAction,10,-1,-1,Others,其他,10/15/2019 19:04,03/04/2025 13:46,Y,1
1,RemedialAction,1,1,1,Crown cleaning,清理樹冠,10/15/2019 19:04,03/04/2025 13:46,Y,2
2,RemedialAction,2,2,2,Crown thinning,樹冠疏理,10/15/2019 19:04,03/04/2025 13:46,Y,3
3,RemedialAction,3,3,3,Crown reduction,縮減樹冠,10/15/2019 19:04,03/04/2025 13:46,Y,4
4,RemedialAction,4,4,4,Crown raising,樹冠提升,10/15/2019 19:04,03/04/2025 13:46,Y,5


In [ ]:
d['TBL_TREE_STATUS.csv'].head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,TreeStatus,1,1,1,Old and Valuable Tree,古樹名木,10/15/2019 18:52,03/04/2025 13:46,Y,1
1,TreeStatus,2,2,2,Stonewall Tree,石牆樹,10/15/2019 18:52,03/04/2025 13:46,Y,2
2,TreeStatus,3,3,3,Brown Root Rot Disease Infected,受褐根病感染,10/15/2019 18:52,03/04/2025 13:46,Y,3
3,TreeStatus,4,4,4,Mature Tree,成齡樹,01/12/2024 01:04,03/04/2025 13:46,N,4
4,TreeStatus,5,5,5,Other Trees,其他樹木,10/15/2019 18:52,03/04/2025 13:46,Y,7


In [ ]:
df_tmcpA.head()

,MasterTypeCode,ID,CSV_CODE,Code,info_eng,info_chi,VERSION_DATETIME,EXPORT_DATETIME,DISPLAY,SORT
0,TreeSpecies,818,1,1,Acacia auriculiformis,耳果相思(耳葉相思),03/06/2020 11:17,03/04/2025 13:46,Y,1
1,TreeSpecies,819,2,2,Acacia confusa,台灣相思,11/24/2015 14:10,03/04/2025 13:46,Y,2
2,TreeSpecies,820,3,3,Acacia dealbata,銀荊,11/24/2015 14:10,03/04/2025 13:46,Y,3
3,TreeSpecies,1541,725,725,Acacia farnesiana,金合歡,02/15/2017 12:06,03/04/2025 13:46,Y,4
4,TreeSpecies,821,4,4,Acacia holosericea,絹毛相思,11/24/2015 14:10,03/04/2025 13:46,Y,5


# read in zip

In [ ]:
f1_summarylist = [
 'form1.ID',
 'form1.ContractGUID',
 'form1.Form1Ref',
 'form1.FileRef',
 'form1.DepartmentGUID',
 'form1.InspectionOfficerGUID',
 'form1.Status',
 'form1.DateOfInspection',
 'form1.LastInspectionTime',
 'form1.InspectionFrequency',
 'form1.LastModifiedTime',
 'form1.LastExportDatetime',
 'form1.LastImportDatetime',
 'form1.MasterZoneGUID',
 'form1.SubZoneGUID',
 'form1.EnglishLocation',
 'form1.ChineseLocation',
 'form1.TreeRiskManagementZone',
 'form1.District',
 'form1.NearestLampPoleNumber',
 'form1.OverallRemarks',
 'form1.TotalNumberOfTreesA',
 'form1.TotalNumberOfTreesB',
 'form1.BlackNumberOfTrees',
 'form1.RedNumberOfTrees',
 'form1.YellowNumberOfTrees',
 'form1.NotClassifiedNumberOfTrees',
 'form1.FileSizeByte',
 'form1.ContractNo',
 'form1.InspectionOfficerFullName',
 'form1.InspectionOfficerNameEnglish',
 'form1.InspectionOfficerNameChinese',
 'form1.InspectionOfficerPost',
 'form1.MasterZoneNumber',
 'form1.SubZoneNumber',
 'form1.OfficerID',
 'form1.MasterZoneID',
 'form1.SubZoneID',
 'form1.Status'
]

## Method 1: read zip in subfolder, return tables in each subfolder

In [ ]:
#folderpath = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4'

d = None
data = None
count = 0
count2 = 0
statusList =[]

#============================================
path = pathlib.Path(folderpath)
print(path)

#===========================================================Methold 1: one time load all zip in folderpath
#files = path.glob("**/*.zip")

#===========================================================Mehtod 2: load each fld first, then load each zip, thrid save xlsx in each fld
loadsubfld = path.glob('*')

for subfld in loadsubfld:
    #print(subfld)

    count = 0
    count2 = 0

    if subfld.is_dir():
        files = subfld.glob("**/202*.zip")
        print(files)

#===========================================================Method2 end
#==========================================================================================

        for zip in files:

            # filter out the unwanted text (unwanted file) in the filepath
            #if str(zip).lower() not in invalidtext:

            checkinvalidlist = []
            for ext in invalidtext:
                checkinvalidlist.append((ext in str(zip).lower()))
            checkvalidlist =[]
            for ext in includetext:
                checkvalidlist.append((ext in str(zip).lower()))
                print('Y')

            #=========================================
            resultA = any(checkinvalidlist)
            resultB = any(checkvalidlist)

            resultA = False
            resultB = True
            #=========================================

            if resultA == False and resultB == True:

                print(zip)
                # zip = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4\20211110160353_HYD-2021-020-8047-0.zip'

                with zipfile.ZipFile(zip, "r") as z:
                    for filename in z.namelist():

                        #===========================================================================================f1
                        if 'json' in filename and 'Version' not in filename:
                            f1 = filename

                            print('form1 : ' + f1)
                            print(Path(zip).parent.absolute())

                            with z.open(f1) as f:
                                json_data = f.read()

                                data = json.loads(json_data)

                                df_summary = pd.json_normalize(data)

                                #====================
                                df_treegp = pd.json_normalize(data, record_path = ['tblForm1TreesBList'])
                                for i in f1_summarylist:
                                    df_treegp[i] = df_summary[i][0]

                                if count == 0:
                                    df_treegpA = pd.DataFrame(columns = df_treegp.columns.tolist())

                                df_treegpA = pd.concat([df_treegp, df_treegpA])
                                df_treegp=None

                                df_treegpA.drop_duplicates()

                                #====================
                                df_works = pd.json_normalize(data, record_path = ['tblForm1TreesList'])
                                for i in f1_summarylist:
                                    df_works[i] = df_summary[i][0]

                                if count == 0:
                                    df_worksA = pd.DataFrame(columns = df_works.columns.tolist())

                                df_worksA = pd.concat([df_works, df_worksA])
                                df_works = None

                                df_worksA.drop_duplicates()

                                #====================
                                df_photo = pd.json_normalize(data, record_path = ['attachment'])
                                for i in f1_summarylist:
                                    df_photo[i] = df_summary[i][0]

                                if count == 0:
                                    df_photoA = pd.DataFrame(columns = df_photo.columns.tolist())

                                df_photoA = pd.concat([df_photo, df_photoA])
                                df_photo=None

                                df_photoA.drop_duplicates()

                                #====================Status
                                status = pd.json_normalize(data)['form1.Status'][0]
                                statusList.append(status)
                                dict = {'status': statusList}
                                df_status = pd.DataFrame(dict)

                                if count == 0:
                                    df_statusA = pd.DataFrame(columns = ['status'])

                                df_statusA = pd.concat([df_status, df_statusA])

                                #=====================
                                if count == 0:
                                    df_summaryA = pd.DataFrame(columns = df_summary.columns.tolist())
                                df_summaryA = pd.concat([df_summary, df_summaryA])
                                df_summary = None

                                #=====================

                                count =  count + 1


                        #===============================================================================================database
                        elif 'json' in filename and 'Version' in filename and '/' not in filename:
                            db = filename

                            with z.open(db) as f:
                                json_data = f.read()

                                data = json.loads(json_data)
                                df_dbsummary = pd.json_normalize(data)

                                #====================
                                df_dbmasterValue = pd.json_normalize(data, record_path = ['masterValueList'])
                                if count2 == 0:
                                    df_dbmasterValueA = pd.DataFrame(columns = df_dbmasterValue.columns.tolist())
                                df_dbmasterValueA = pd.concat([df_dbmasterValue, df_dbmasterValueA])
                                df_dbmasterValueA.drop_duplicates()

                                #====================
                                df_dbtreeList = pd.json_normalize(data, record_path = ['treeList'])
                                if count2 == 0:
                                    df_dbtreeListA = pd.DataFrame(columns = df_dbtreeList.columns.tolist())
                                df_dbtreeListA = pd.concat([df_dbtreeList, df_dbtreeListA])
                                df_dbtreeList = None

                                df_dbtreeListA.drop_duplicates()

                                #=====================
                                count2 = count2 +1

#=====================================the below is the by-product of the programming============================================================================================
#==============================================================================================================================================lookup tmcplist vs df_db series
        # misleading fix
        df_dbmasterValueA.MasterTypeCode = df_dbmasterValueA.MasterTypeCode.astype(str)
        df_dbmasterValueA.Code = df_dbmasterValueA.Code.astype(str)
        df_tmcpA.MasterTypeCode = df_tmcpA.MasterTypeCode.astype(str)
        df_tmcpA.Code = df_tmcpA.Code.astype(str)

        # merge/concat
        df_decode = pd.merge(df_dbmasterValueA, df_tmcpA, how='left', left_on=['MasterTypeCode', 'Code'], right_on = ['MasterTypeCode','Code'])
        df_decode

        #========================================================================================
        # Decode by replace

        df_worksA2 = df_worksA.replace(df_decode.ID_x.tolist(), df_decode.Code.tolist())
        df_worksAdecode = df_worksA2.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

        #format date
        df_worksAdecode['form1.DateOfInspection'] = pd.to_datetime(df_worksAdecode['form1.DateOfInspection'])
        df_worksAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_worksAdecode['form1.LastInspectionTime'])

        #========================================================================================
        # Decode by replace
        df_treegpAdecode = df_treegpA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

        #format date
        df_treegpAdecode['form1.DateOfInspection'] = pd.to_datetime(df_treegpAdecode['form1.DateOfInspection'])
        df_treegpAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_treegpAdecode['form1.LastInspectionTime'])

        #========================================================================================
        # Decode by replace
        df_photoAdecode = df_photoA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

        #format date
        df_photoAdecode['form1.DateOfInspection'] = pd.to_datetime(df_photoAdecode['form1.DateOfInspection'])
        df_photoAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_photoAdecode['form1.LastInspectionTime'])

        #========================================================================================

        #===================================================================================================================================================
        #===========================================================================================summary list preaparation
        droplist = ['attachment',
         'tblForm1TreesList',
         'tblLocationTypeList',
         'tblForm1TreesBList',
         'tblForm1AuditCommentList',
         'photoList',
         'auditTabs']


        df_summaryALookup = df_summaryA.replace(df_decode.ID_x.tolist(), df_decode.Code.tolist())

        df_summaryADrop = df_summaryALookup.drop(droplist, axis=1)

        #format date
        df_summaryADrop['form1.DateOfInspection'] = pd.to_datetime(df_summaryADrop['form1.DateOfInspection'])
        df_summaryADrop['form1.LastInspectionTime'] = pd.to_datetime(df_summaryADrop['form1.LastInspectionTime'])
        #====================================================================================================================================

        #========================================================================================print to xl
        wb = xw.Book()

        wb.sheets[0].range("A1").value =  df_worksAdecode
        wb.sheets[0].range("D:D, I:K, AF:AF, AI:AJ, AR:AW, BA:BB").color = (255,255,204)
        wb.sheets[0].name = 'Mitigation'

        wb.sheets.add()
        wb.sheets[0].range("A1").value =  df_treegpAdecode
        wb.sheets[0].range("D:H, Q:R, L:L, AN:AN, AQ:AR").color = (255,255,204)
        wb.sheets[0].name = 'TreeGp'

        wb.sheets.add()
        wb.sheets[0].range("A1").value =  df_photoAdecode
        wb.sheets[0].range("E:E, H:H, J:J, M:M, R:S, Z:Z, AO:AS").color = (255,255,204)
        wb.sheets[0].name = 'Photo'

        wb.sheets.add()
        wb.sheets[0].range("A1").value =  df_summaryADrop
        wb.sheets[0].range("D:D, I:J, W:AB, AF:AF, AI:AJ").color = (255,255,204)
        wb.sheets[0].name = 'Summary'

        for ws in wb.sheets:
            ws.autofit(axis="columns")

        #============================================================================save
        #wb.save(Path('/').joinpath(folderpath, Path(folderpath).name + '_summary.xlsx'))
        wb.save(Path('/').joinpath(subfld, Path(subfld).name + '_summary.xlsx'))

        #============================================================================clear dataframe
        df_treegpA = None
        df_photoA = None
        df_dbsummary = None
        df_summary = None
        df_worksA = None
        df_summaryALookup = None
        df_dbmasterValueA = None
        df_dbtreeListA = None
        df_worksA2 = None
print('End')

H:\LU\TRAM 10 Form 1 (2024)\Submission Report
<generator object Path.glob at 0x00000261CE348D60>
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC246_0(0)\20241126163155_HYD-2024-717-1393-0.zip


C:\Users\ckho\AppData\Local\Temp\ipykernel_16852\1044315474.py:157: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_dbtreeListA = pd.concat([df_dbtreeList, df_dbtreeListA])


form1 : F1_HYD-2024-717-1393-0/HYD-2024-717-1393-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC246_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC247_0(0)\20241126163406_HYD-2024-717-1391-0.zip


C:\Users\ckho\AppData\Local\Temp\ipykernel_16852\1044315474.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_treegpA = pd.concat([df_treegp, df_treegpA])
C:\Users\ckho\AppData\Local\Temp\ipykernel_16852\1044315474.py:96: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_worksA = pd.concat([df_works, df_worksA])


form1 : F1_HYD-2024-717-1391-0/HYD-2024-717-1391-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC247_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC251_0(0)\20241127143917_HYD-2024-717-1590-0.zip
form1 : F1_HYD-2024-717-1590-0/HYD-2024-717-1590-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC251_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC358_0(0)\20241127143917_HYD-2024-717-1397-0.zip
form1 : F1_HYD-2024-717-1397-0/HYD-2024-717-1397-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC358_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC436_0(0)\20241127143917_HYD-2024-717-1388-0.zip
form1 : F1_HYD-2024-717-1388-0/HYD-2024-717-1388-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125_NP\unzip\NP_11SE-AC436_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\1 20241125

C:\Users\ckho\AppData\Local\Temp\ipykernel_16852\1044315474.py:157: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_dbtreeListA = pd.concat([df_dbtreeList, df_dbtreeListA])
C:\Users\ckho\AppData\Local\Temp\ipykernel_16852\1044315474.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_treegpA = pd.concat([df_treegp, df_treegpA])


form1 : F1_HYD-2024-717-2303/HYD-2024-717-2303.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicate with Batch 7_no need to submit\SW_11SW-DCR1938_2(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicate with batch 8 (Part 2)_no need to submit\SE_15NE-AC182_0(0)\SE_15NE-AC182_0(0)\20250212185703_HYD-2024-717-0190-0.zip


C:\Users\ckho\AppData\Local\Temp\ipykernel_16852\1044315474.py:96: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_worksA = pd.concat([df_works, df_worksA])


form1 : F1_HYD-2024-717-0190-0/HYD-2024-717-0190-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicate with batch 8 (Part 2)_no need to submit\SE_15NE-AC182_0(0)\SE_15NE-AC182_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicate with batch 8 (Part 2)_no need to submit\SE_15NE-AC192_0(0)\SE_15NE-AC192_0(0)\20250212191418_HYD-2024-717-2261-0.zip
form1 : F1_HYD-2024-717-2261-0/HYD-2024-717-2261-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicate with batch 8 (Part 2)_no need to submit\SE_15NE-AC192_0(0)\SE_15NE-AC192_0(0)
Y
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicate with batch 8 (Part 2)_no need to submit\SE_15NE-AC643_1(0)\SE_15NE-AC643_1(0)\20250212185703_HYD-2024-717-2652-0.zip
form1 : F1_HYD-2024-717-2652-0/HYD-2024-717-2652-0.json
H:\LU\TRAM 10 Form 1 (2024)\Submission Report\10 20250228_W, SW, SE, P, NP, E\Duplicat

In [ ]:
df_worksAdecode.columns.to_list()

## Method 1 : read zips in subfolder, return table to a parent folder

In [ ]:
#folderpath = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4'

d = None
data = None
count = 0
count2 = 0

#============================================
path = pathlib.Path(folderpath)
print(path)

#===========================================================Methold 1: one time load all zip in folderpath
files = path.glob("**/*.zip")

#===========================================================Mehtod 2: load each fld first, then load each zip, thrid save xlsx in each fld
#loadsubfld = path.glob('*')

#for subfld in loadsubfld:
    #print(subfld)

    #count = 0
    #count2 = 0

    #if subfld.is_dir():
        #files = subfld.glob("**/*.zip")
        #print(files)

#===========================================================Method2 end
#==========================================================================================

for zip in files:

    # filter out the unwanted text (unwanted file) in the filepath
    #if str(zip).lower() not in invalidtext:

    checkinvalidlist = []
    for ext in invalidtext:
        checkinvalidlist.append((ext in str(zip).lower()))
    checkvalidlist =[]
    for ext in includetext:
        checkvalidlist.append((ext in str(zip).lower()))
        print('Y')

    if any(checkinvalidlist) == False and any(checkvalidlist) == True:

        print(zip)
        # zip = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4\20211110160353_HYD-2021-020-8047-0.zip'

        with zipfile.ZipFile(zip, "r") as z:
            for filename in z.namelist():

                #===========================================================================================f1
                if 'json' in filename and 'Version' not in filename:
                    f1 = filename

                    print('form1 :' + f1)
                    print(Path(zip).parent.absolute())

                    with z.open(f1) as f:
                        json_data = f.read()

                        data = json.loads(json_data)

                        df_summary = pd.json_normalize(data)

                        #====================
                        df_treegp = pd.json_normalize(data, record_path = ['tblForm1TreesBList'])
                        for i in f1_summarylist:
                            df_treegp[i] = df_summary[i][0]

                        if count == 0:
                            df_treegpA = pd.DataFrame(columns = df_treegp.columns.tolist())

                        df_treegpA = pd.concat([df_treegp, df_treegpA])
                        df_treegp=None

                        df_treegpA.drop_duplicates()

                        #====================
                        df_works = pd.json_normalize(data, record_path = ['tblForm1TreesList'])
                        for i in f1_summarylist:
                            df_works[i] = df_summary[i][0]

                        if count == 0:
                            df_worksA = pd.DataFrame(columns = df_works.columns.tolist())

                        df_worksA = pd.concat([df_works, df_worksA])
                        df_works = None

                        df_worksA.drop_duplicates()

                        #====================
                        df_photo = pd.json_normalize(data, record_path = ['attachment'])
                        for i in f1_summarylist:
                            df_photo[i] = df_summary[i][0]

                        if count == 0:
                            df_photoA = pd.DataFrame(columns = df_photo.columns.tolist())

                        df_photoA = pd.concat([df_photo, df_photoA])
                        df_photo=None

                        df_photoA.drop_duplicates()

                        #=====================
                        if count == 0:
                            df_summaryA = pd.DataFrame(columns = df_summary.columns.tolist())
                        df_summaryA = pd.concat([df_summary, df_summaryA])
                        df_summary = None

                        #=====================

                        count =  count + 1


                #===============================================================================================database
                elif 'json' in filename and 'Version' in filename and '/' not in filename:
                    db = filename

                    with z.open(db) as f:
                        json_data = f.read()

                        data = json.loads(json_data)
                        df_dbsummary = pd.json_normalize(data)

                        #====================
                        df_dbmasterValue = pd.json_normalize(data, record_path = ['masterValueList'])
                        if count2 == 0:
                            df_dbmasterValueA = pd.DataFrame(columns = df_dbmasterValue.columns.tolist())
                        df_dbmasterValueA = pd.concat([df_dbmasterValue, df_dbmasterValueA])
                        df_dbmasterValueA.drop_duplicates()

                        #====================
                        df_dbtreeList = pd.json_normalize(data, record_path = ['treeList'])
                        if count2 == 0:
                            df_dbtreeListA = pd.DataFrame(columns = df_dbtreeList.columns.tolist())
                        df_dbtreeListA = pd.concat([df_dbtreeList, df_dbtreeListA])
                        df_dbtreeList = None

                        df_dbtreeListA.drop_duplicates()

                        count2 = count2 +1

#=====================================the below is the by-product of the programming============================================================================================
#==============================================================================================================================================lookup tmcplist vs df_db series
# misleading fix
df_dbmasterValueA.MasterTypeCode = df_dbmasterValueA.MasterTypeCode.astype(str)
df_dbmasterValueA.Code = df_dbmasterValueA.Code.astype(str)
df_tmcpA.MasterTypeCode = df_tmcpA.MasterTypeCode.astype(str)
df_tmcpA.Code = df_tmcpA.Code.astype(str)

# merge/concat
df_decode = pd.merge(df_dbmasterValueA, df_tmcpA, how='left', left_on=['MasterTypeCode', 'Code'], right_on = ['MasterTypeCode','Code'])
df_decode

#========================================================================================
# Decode by replace
df_worksAdecode = df_worksA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_worksAdecode['form1.DateOfInspection'] = pd.to_datetime(df_worksAdecode['form1.DateOfInspection'])
df_worksAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_worksAdecode['form1.LastInspectionTime'])

#========================================================================================
# Decode by replace
df_treegpAdecode = df_treegpA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_treegpAdecode['form1.DateOfInspection'] = pd.to_datetime(df_treegpAdecode['form1.DateOfInspection'])
df_treegpAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_treegpAdecode['form1.LastInspectionTime'])

#========================================================================================
# Decode by replace
df_photoAdecode = df_photoA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_photoAdecode['form1.DateOfInspection'] = pd.to_datetime(df_photoAdecode['form1.DateOfInspection'])
df_photoAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_photoAdecode['form1.LastInspectionTime'])


#===================================================================================================================================================
#===========================================================================================summary list preaparation
droplist = ['attachment',
 'tblForm1TreesList',
 'tblLocationTypeList',
 'tblForm1TreesBList',
 'tblForm1AuditCommentList',
 'photoList',
 'auditTabs']

df_summaryADrop = df_summaryA.drop(droplist, axis=1)

#format date
df_summaryADrop['form1.DateOfInspection'] = pd.to_datetime(df_summaryADrop['form1.DateOfInspection'])
df_summaryADrop['form1.LastInspectionTime'] = pd.to_datetime(df_summaryADrop['form1.LastInspectionTime'])
#====================================================================================================================================

#========================================================================================print to xl
wb = xw.Book()

wb.sheets[0].range("A1").value =  df_worksAdecode
wb.sheets[0].range("D:D, I:K, AF:AF, AI:AJ, AR:AW, BA:BB").color = (255,255,204)
wb.sheets[0].name = 'Mitigation'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_treegpAdecode
wb.sheets[0].range("D:H, Q:R, L:L, AN:AN, AQ:AR").color = (255,255,204)
wb.sheets[0].name = 'TreeGp'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_photoAdecode
wb.sheets[0].range("E:E, H:H, J:J, M:M, R:S, Z:Z, AO:AS").color = (255,255,204)
wb.sheets[0].name = 'Photo'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_summaryADrop
wb.sheets[0].range("D:D, I:J, W:AB, AF:AF, AI:AJ").color = (255,255,204)
wb.sheets[0].name = 'Summary'

for ws in wb.sheets:
    ws.autofit(axis="columns")

#============================================================================save
wb.save(Path('/').joinpath(folderpath, Path(folderpath).name + '_summary.xlsx'))
#wb.save(Path('/').joinpath(subfld, Path(subfld).name + '_summary.xlsx'))

print('completed')
#============================================================================clear dataframe
df_treegpA = None
df_photoA = None
df_dbsummary = None
df_summary = None
df_worksA = None
df_dbmasterValueA = None
df_dbtreeListA = None

## Method 2: Open zip within zip

In [ ]:
#folderpath = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4'

d = None
data = None
count = 0
count2 = 0

#============================================
path = pathlib.Path(folderpath)
print(path)

#===========================================================Methold 1: one time load all zip in folderpath
files = path.glob("**/*.zip")

#===========================================================Mehtod 2: load each fld first, then load each zip, thrid save xlsx in each fld
#loadsubfld = path.glob('*')

#for subfld in loadsubfld:
    #print(subfld)

    #count = 0
    #count2 = 0

    #if subfld.is_dir():
        #files = subfld.glob("**/*.zip")
        #print(files)

#===========================================================Method2 end
#==========================================================================================

for zip in files:
    print(zip)

    # filter out the unwanted text (unwanted file) in the filepath
    #if str(zip).lower() not in invalidtext:

    checkinvalidlist = []
    for ext in invalidtext:
        checkinvalidlist.append((ext in str(zip).lower()))
    checkvalidlist =[]
    for ext in includetext:
        checkvalidlist.append((ext in str(zip).lower()))
        print('Y')

    if any(checkinvalidlist) == False and any(checkvalidlist) == True:

        print(zip)
        # zip = r'L:\LU\Staff\Ricky\---System Checking\TMCP\1. Raw data\202111\raw\TMCP 202111-E\week4\20211110160353_HYD-2021-020-8047-0.zip'

        with zipfile.ZipFile(zip, "r") as outer_zip:
            for file in outer_zip.namelist():
                print(f'\nouterfile: {file}')

                if file.endswith('.zip'):


                    #====
                    with outer_zip.open(file) as inner_zip_file:
                        # Read the inner zip file
                        with zipfile.ZipFile(inner_zip_file) as inner_zip:
                            inner_files = inner_zip.namelist()
                            for inner_file in inner_files:
                                print(f'\ninnerfile: {inner_file}')

                                #===========================================================================================f1
                                if 'json' in inner_file and 'Version' not in inner_file:
                                    f1 = inner_file

                                    print(f'\nform1: {f1}')
                                    print(Path(zip).parent.absolute())

                                    with inner_zip.open(f1) as f:
                                        json_data = f.read()

                                        data = json.loads(json_data)
                                        print(data)

                                        df_summary = pd.json_normalize(data)

                                        #====================
                                        df_treegp = pd.json_normalize(data, record_path = ['tblForm1TreesBList'])
                                        print(df_treegp)
                                        for i in f1_summarylist:
                                            df_treegp[i] = df_summary[i][0]

                                        if count == 0:
                                            df_treegpA = pd.DataFrame(columns = df_treegp.columns.tolist())

                                        df_treegpA = pd.concat([df_treegp, df_treegpA])
                                        df_treegp=None

                                        df_treegpA.drop_duplicates()

                                        #====================
                                        df_works = pd.json_normalize(data, record_path = ['tblForm1TreesList'])
                                        for i in f1_summarylist:
                                            df_works[i] = df_summary[i][0]

                                        if count == 0:
                                            df_worksA = pd.DataFrame(columns = df_works.columns.tolist())

                                        df_worksA = pd.concat([df_works, df_worksA])
                                        df_works = None

                                        df_worksA.drop_duplicates()

                                        #====================
                                        df_photo = pd.json_normalize(data, record_path = ['attachment'])
                                        for i in f1_summarylist:
                                            df_photo[i] = df_summary[i][0]

                                        if count == 0:
                                            df_photoA = pd.DataFrame(columns = df_photo.columns.tolist())

                                        df_photoA = pd.concat([df_photo, df_photoA])
                                        df_photo=None

                                        df_photoA.drop_duplicates()

                                        #=====================
                                        if count == 0:
                                            df_summaryA = pd.DataFrame(columns = df_summary.columns.tolist())
                                        df_summaryA = pd.concat([df_summary, df_summaryA])
                                        df_summary = None

                                        #=====================

                                        count =  count + 1


                                #===============================================================================================database
                                elif 'json' in inner_file and 'Version' in inner_file and '/' not in inner_file:
                                    db = inner_file

                                    #with z.open(db) as f:
                                    with inner_zip.open(db) as f:
                                        json_data = f.read()

                                        data = json.loads(json_data)
                                        df_dbsummary = pd.json_normalize(data)

                                        #====================
                                        df_dbmasterValue = pd.json_normalize(data, record_path = ['masterValueList'])
                                        if count2 == 0:
                                            df_dbmasterValueA = pd.DataFrame(columns = df_dbmasterValue.columns.tolist())
                                        df_dbmasterValueA = pd.concat([df_dbmasterValue, df_dbmasterValueA])
                                        df_dbmasterValueA.drop_duplicates()

                                        #====================
                                        df_dbtreeList = pd.json_normalize(data, record_path = ['treeList'])
                                        if count2 == 0:
                                            df_dbtreeListA = pd.DataFrame(columns = df_dbtreeList.columns.tolist())
                                        df_dbtreeListA = pd.concat([df_dbtreeList, df_dbtreeListA])
                                        df_dbtreeList = None

                                        df_dbtreeListA.drop_duplicates()

                                        count2 = count2 +1

#=====================================the below is the by-product of the programming============================================================================================
#==============================================================================================================================================lookup tmcplist vs df_db series
# misleading fix
df_dbmasterValueA.MasterTypeCode = df_dbmasterValueA.MasterTypeCode.astype(str)
df_dbmasterValueA.Code = df_dbmasterValueA.Code.astype(str)
df_tmcpA.MasterTypeCode = df_tmcpA.MasterTypeCode.astype(str)
df_tmcpA.Code = df_tmcpA.Code.astype(str)

# merge/concat
df_decode = pd.merge(df_dbmasterValueA, df_tmcpA, how='left', left_on=['MasterTypeCode', 'Code'], right_on = ['MasterTypeCode','Code'])
df_decode

#========================================================================================
# Decode by replace
df_worksAdecode = df_worksA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_worksAdecode['form1.DateOfInspection'] = pd.to_datetime(df_worksAdecode['form1.DateOfInspection'])
df_worksAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_worksAdecode['form1.LastInspectionTime'])

#========================================================================================
# Decode by replace
df_treegpAdecode = df_treegpA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_treegpAdecode['form1.DateOfInspection'] = pd.to_datetime(df_treegpAdecode['form1.DateOfInspection'])
df_treegpAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_treegpAdecode['form1.LastInspectionTime'])

#========================================================================================
# Decode by replace
df_photoAdecode = df_photoA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_photoAdecode['form1.DateOfInspection'] = pd.to_datetime(df_photoAdecode['form1.DateOfInspection'])
df_photoAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_photoAdecode['form1.LastInspectionTime'])


#===================================================================================================================================================
#===========================================================================================summary list preaparation
droplist = ['attachment',
 'tblForm1TreesList',
 'tblLocationTypeList',
 'tblForm1TreesBList',
 'tblForm1AuditCommentList',
 'photoList',
 'auditTabs']

df_summaryADrop = df_summaryA.drop(droplist, axis=1)

#format date
df_summaryADrop['form1.DateOfInspection'] = pd.to_datetime(df_summaryADrop['form1.DateOfInspection'])
df_summaryADrop['form1.LastInspectionTime'] = pd.to_datetime(df_summaryADrop['form1.LastInspectionTime'])
#====================================================================================================================================

#========================================================================================print to xl
wb = xw.Book()

wb.sheets[0].range("A1").value =  df_worksAdecode
wb.sheets[0].range("D:D, I:K, AF:AF, AI:AJ, AR:AW, BA:BB").color = (255,255,204)
wb.sheets[0].name = 'Mitigation'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_treegpAdecode
wb.sheets[0].range("D:H, Q:R, L:L, AN:AN, AQ:AR").color = (255,255,204)
wb.sheets[0].name = 'TreeGp'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_photoAdecode
wb.sheets[0].range("E:E, H:H, J:J, M:M, R:S, Z:Z, AO:AS").color = (255,255,204)
wb.sheets[0].name = 'Photo'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_summaryADrop
wb.sheets[0].range("D:D, I:J, W:AB, AF:AF, AI:AJ").color = (255,255,204)
wb.sheets[0].name = 'Summary'

for ws in wb.sheets:
    ws.autofit(axis="columns")

#============================================================================save
wb.save(Path('/').joinpath(folderpath, Path(folderpath).name + '_summary.xlsx'))
#wb.save(Path('/').joinpath(subfld, Path(subfld).name + '_summary.xlsx'))

#============================================================================clear dataframe
df_treegpA = None
df_photoA = None
df_dbsummary = None
df_summary = None
df_worksA = None
df_dbmasterValueA = None
df_dbtreeListA = None

In [ ]:
df_dbmasterValueA.MasterTypeCode = df_dbmasterValueA.MasterTypeCode.astype(str)
df_dbmasterValueA.Code = df_dbmasterValueA.Code.astype(str)
df_tmcpA.MasterTypeCode = df_tmcpA.MasterTypeCode.astype(str)
df_tmcpA.Code = df_tmcpA.Code.astype(str)

# merge/concat
df_decode = pd.merge(df_dbmasterValueA, df_tmcpA, how='left', left_on=['MasterTypeCode', 'Code'], right_on = ['MasterTypeCode','Code'])
df_decode

#========================================================================================
# Decode by replace
df_worksAdecode = df_worksA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_worksAdecode['form1.DateOfInspection'] = pd.to_datetime(df_worksAdecode['form1.DateOfInspection'])
df_worksAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_worksAdecode['form1.LastInspectionTime'])

#========================================================================================
# Decode by replace
df_treegpAdecode = df_treegpA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_treegpAdecode['form1.DateOfInspection'] = pd.to_datetime(df_treegpAdecode['form1.DateOfInspection'])
df_treegpAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_treegpAdecode['form1.LastInspectionTime'])

#========================================================================================
# Decode by replace
df_photoAdecode = df_photoA.replace(df_decode.ID_x.tolist(), df_decode.info_eng.tolist())

#format date
df_photoAdecode['form1.DateOfInspection'] = pd.to_datetime(df_photoAdecode['form1.DateOfInspection'])
df_photoAdecode['form1.LastInspectionTime'] = pd.to_datetime(df_photoAdecode['form1.LastInspectionTime'])


#===================================================================================================================================================
#===========================================================================================summary list preaparation
droplist = ['attachment',
 'tblForm1TreesList',
 'tblLocationTypeList',
 'tblForm1TreesBList',
 'tblForm1AuditCommentList',
 'photoList',
 'auditTabs']

df_summaryADrop = df_summaryA.drop(droplist, axis=1)

#format date
df_summaryADrop['form1.DateOfInspection'] = pd.to_datetime(df_summaryADrop['form1.DateOfInspection'])
df_summaryADrop['form1.LastInspectionTime'] = pd.to_datetime(df_summaryADrop['form1.LastInspectionTime'])
#====================================================================================================================================

#========================================================================================print to xl
wb = xw.Book()

wb.sheets[0].range("A1").value =  df_worksAdecode
wb.sheets[0].range("D:D, I:K, AF:AF, AI:AJ, AR:AW, BA:BB").color = (255,255,204)
wb.sheets[0].name = 'Mitigation'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_treegpAdecode
wb.sheets[0].range("D:H, Q:R, L:L, AN:AN, AQ:AR").color = (255,255,204)
wb.sheets[0].name = 'TreeGp'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_photoAdecode
wb.sheets[0].range("E:E, H:H, J:J, M:M, R:S, Z:Z, AO:AS").color = (255,255,204)
wb.sheets[0].name = 'Photo'

wb.sheets.add()
wb.sheets[0].range("A1").value =  df_summaryADrop
wb.sheets[0].range("D:D, I:J, W:AB, AF:AF, AI:AJ").color = (255,255,204)
wb.sheets[0].name = 'Summary'

for ws in wb.sheets:
    ws.autofit(axis="columns")

#============================================================================save
wb.save(Path('/').joinpath(folderpath, Path(folderpath).name + '_summary.xlsx'))
#wb.save(Path('/').joinpath(subfld, Path(subfld).name + '_summary.xlsx'))

print('completed')
#============================================================================clear dataframe
df_treegpA = None
df_photoA = None
df_dbsummary = None
df_summary = None
df_worksA = None
df_dbmasterValueA = None
df_dbtreeListA = None

In [ ]:
data

In [ ]:
filename

In [ ]:
df_dbmasterValue = pd.json_normalize(data, record_path = ['masterValueList'])

In [ ]:
json_data

In [ ]:
d = None
data = None
count = 0
count2 = 0

#============================================
path = pathlib.Path(folderpath)
print(path)

#===========================================================Methold 1: one time load all zip in folderpath
#files = path.glob("**/*.zip")

#===========================================================Mehtod 2: load each fld first, then load each zip, thrid save xlsx in each fld
loadsubfld = path.glob('*')

for subfld in loadsubfld:
    #print(subfld)

    count = 0
    count2 = 0

    #if subfld.is_dir():
    files = subfld.glob("**/*.zip")
    #print(files)

#===========================================================Method2 end
#==========================================================================================

    for zip in files:
        print(zip)

In [ ]:
files

# below is the by-product of the programming

## lookup tmcplist vs df_db series

## Decode and dateformatting